In [ ]:
import os
import sys
import warnings
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

# Élimination des avertissements
if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore"

In [ ]:
# Stopwords & punctuation
import string
from nltk.corpus import stopwords
punct = string.punctuation
stopw = stopwords.words('english') + [x for x in punct]
stopw += [x.translate
    (str.maketrans('', '', punct)) for x in stopwords.words('english')]

stopw +=  ["'d", "'ll", "'re", "'s", "'ve", '``', 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would']

# Tokenizer
from nltk.tokenize import word_tokenize

def tokenize_remove_stopwords(text):
 for token in word_tokenize(text):
    if token in stopw: continue
    yield (token)

**Lecture des données**

In [ ]:
# Lecture du jeu de données et séparation de celles-ci en ensembles d'entraînement et de test

train = pd.read_excel('../1-data/training_datasets/train_dataset_40pc.xlsx')
test = pd.read_excel('../1-data/test_dataset_10pc.xlsx')

**Entraînement des modèles**

In [ ]:
X_train, y_train = train.text_post.astype('str'), train.category
X_test, y_test = test.text_post.astype('str'), test.category

In [ ]:
# Définition du pipeline
n_features = [100, 250, 500, 750, 1000, 2500, 5000, 10000, 15000]

pipeline = Pipeline(
    [
        ("vectorizer", TfidfVectorizer(            
            stop_words=stopw,
            tokenizer=word_tokenize,
            token_pattern=None)),
        ("classify", "passthrough"),
    ]
)

param_grid = [
    {
        "vectorizer__max_features": n_features,
        "classify" : [
            LogisticRegression(n_jobs=3), 
            LinearSVC(dual="auto"),
            MultinomialNB(),
            RandomForestClassifier(n_jobs=3)
            ]
    }
]

In [ ]:
grid_search = GridSearchCV(
    pipeline, 
    param_grid=param_grid, 
    n_jobs=2, 
    verbose=1, 
    refit='f1_macro', 
    scoring=['accuracy','f1_macro']
)

In [ ]:
grid_search.fit(X_train, y_train)

In [ ]:
grid_search.best_params_

In [ ]:
results_cv = pd.DataFrame(grid_search.cv_results_)
results_cv = results_cv[
    ['param_classify', 'param_vectorizer__max_features', 
     'split0_test_accuracy', 'split1_test_accuracy', 'split2_test_accuracy',
       'split3_test_accuracy', 'split4_test_accuracy', 'mean_test_accuracy',
       'std_test_accuracy', 'rank_test_accuracy', 'split0_test_f1_macro',
       'split1_test_f1_macro', 'split2_test_f1_macro', 'split3_test_f1_macro',
       'split4_test_f1_macro', 'mean_test_f1_macro', 'std_test_f1_macro',
       'rank_test_f1_macro']
    ]

results_cv.sort_values(by='rank_test_f1_macro')